# Module 7 | Class 3 -- Sentiment Analysis with TF-IDF + Logistic Regression

**Objective:** Build a complete text classification pipeline from raw text to evaluation metrics.

Steps:
1. Load and explore a text dataset
2. Preprocess text
3. Convert text to TF-IDF features
4. Train a Logistic Regression classifier
5. Evaluate with accuracy, precision, recall, F1, and confusion matrix
6. Inspect top positive/negative words

## 1. Setup and Data Loading

We will use a subset of Amazon product reviews. The dataset is fetched directly from a public URL.

In [ ]:
!pip install -q nltk scikit-learn pandas matplotlib

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords

RANDOM_STATE = 42
print("Setup complete.")

### Load Dataset

We use a labeled Amazon reviews dataset (positive/negative). If the download fails, we fall back to sklearn's 20newsgroups.

In [ ]:
# Try Amazon reviews first, fall back to 20newsgroups
try:
    url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv"
    df = pd.read_csv(url)
    # This dataset has 'reviewText' and 'Positive' columns
    df = df.rename(columns={'reviewText': 'text', 'Positive': 'label'})
    df = df[['text', 'label']].dropna()
    print(f"Loaded Amazon reviews: {len(df)} rows")
    data_source = "amazon"
except Exception as e:
    print(f"Amazon download failed ({e}), using 20newsgroups fallback.")
    from sklearn.datasets import fetch_20newsgroups
    # Use 2 categories as a binary classification proxy
    cats = ['rec.sport.baseball', 'sci.space']
    raw = fetch_20newsgroups(subset='all', categories=cats, remove=('headers', 'footers', 'quotes'))
    df = pd.DataFrame({'text': raw.data, 'label': raw.target})
    df = df[df['text'].str.strip().str.len() > 20]
    print(f"Loaded 20newsgroups ({cats}): {len(df)} rows")
    data_source = "newsgroups"

print(f"\nLabel distribution:\n{df['label'].value_counts()}")
df.head()

## 2. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
# Keep negation words -- we learned in Class 1 why this matters
negation_words = {'not', 'no', 'nor', 'never', 'neither', "don't", "doesn't", "didn't",
                  "won't", "wouldn't", "couldn't", "shouldn't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negation_words

def clean_text(text):
    """Lowercase, remove punctuation, remove stopwords (keep negations)."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

df['clean'] = df['text'].apply(clean_text)

print("Original:", df['text'].iloc[0][:120])
print("Cleaned: ", df['clean'].iloc[0][:120])

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, random_state=RANDOM_STATE, stratify=df['label']
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 4. TF-IDF Vectorization

TF-IDF (Term Frequency - Inverse Document Frequency) converts text into numerical feature vectors.
Words that appear frequently in one document but rarely across all documents get higher weights.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Sample features: {list(tfidf.vocabulary_.keys())[:10]}")

## 5. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print("Training complete.")

## 6. Evaluation

In [ ]:
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("=" * 50)
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Top Positive and Negative Words

Logistic regression coefficients tell us which words push the prediction toward each class.
Higher positive coefficients = stronger signal for class 1 (positive).
Lower (most negative) coefficients = stronger signal for class 0 (negative).

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())

# For binary classification, coefficients are in model.coef_[0]
coefficients = model.coef_[0]

# Top 15 positive and negative
top_positive_idx = np.argsort(coefficients)[-15:][::-1]
top_negative_idx = np.argsort(coefficients)[:15]

print("Top 15 POSITIVE words (push toward label=1):")
print("-" * 40)
for idx in top_positive_idx:
    print(f"  {feature_names[idx]:25s} {coefficients[idx]:+.4f}")

print()
print("Top 15 NEGATIVE words (push toward label=0):")
print("-" * 40)
for idx in top_negative_idx:
    print(f"  {feature_names[idx]:25s} {coefficients[idx]:+.4f}")

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names[top_positive_idx][::-1], coefficients[top_positive_idx][::-1], color='green')
axes[0].set_title('Top Positive Words')

axes[1].barh(feature_names[top_negative_idx][::-1], coefficients[top_negative_idx][::-1], color='red')
axes[1].set_title('Top Negative Words')

plt.tight_layout()
plt.show()

## 8. Quick Predictions on New Text

In [ ]:
def predict_sentiment(text):
    """Clean text, vectorize, predict, and return label + probability."""
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    label = "Positive" if pred == 1 else "Negative"
    return label, prob

samples = [
    "This product is amazing, best purchase I've ever made!",
    "Terrible quality, broke after one day. Total waste of money.",
    "It's okay, nothing special but does the job.",
    "Absolutely love it, exceeded all my expectations.",
    "DO NOT buy this, worst experience ever."
]

for s in samples:
    label, prob = predict_sentiment(s)
    print(f"[{label:8s}] (conf: {max(prob):.2f}) | {s}")

---
## TODO: Student Exercise

Experiment with the pipeline by changing parameters and reporting results.

**Task 1:** Change TF-IDF parameters and retrain:
- Try `max_features=2000` (fewer features)
- Try `max_features=10000` (more features)
- Try `ngram_range=(1, 1)` (unigrams only)
- For each, report accuracy and F1 score.

**Task 2:** Change LogisticRegression `C` parameter:
- Try `C=0.01`, `C=0.1`, `C=1.0`, `C=10.0`
- Report accuracy and F1 for each.

**Task 3:** Write 3 of your own sentences and predict their sentiment.
Include at least one tricky case (sarcasm, negation, mixed).

In [ ]:
# TODO: Task 1 -- Different TF-IDF parameters
# Example structure:
#   tfidf_v2 = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
#   X_train_v2 = tfidf_v2.fit_transform(X_train)
#   X_test_v2 = tfidf_v2.transform(X_test)
#   model_v2 = LogisticRegression(max_iter=1000, random_state=42)
#   model_v2.fit(X_train_v2, y_train)
#   y_pred_v2 = model_v2.predict(X_test_v2)
#   print(f"Accuracy: {accuracy_score(y_test, y_pred_v2):.4f}")
#   print(f"F1:       {f1_score(y_test, y_pred_v2, average='weighted'):.4f}")

# Your code here


In [ ]:
# TODO: Task 2 -- Different C values

# Your code here


In [ ]:
# TODO: Task 3 -- Your own sentences

# Your code here
